### SkillFormer Ego

Overall test accuracy: 0.4454  

Accuracy by scenario:  
  Basketball: 0.4545 (55 samples)  
  Bouldering: 0.2658 (79 samples)  
  Cooking: 0.4167 (12 samples)  
  Dance: 0.5510 (49 samples)  
  Music: 0.7857 (28 samples)  
  Soccer: 0.3333 (6 samples)  

Overall predicted class distribution:  
  Class 0: 20 samples (8.73%)  
  Class 1: 86 samples (37.55%)  
  Class 2: 54 samples (23.58%)  
  Class 3: 69 samples (30.13%)  

### Skillformer Exo

Overall test accuracy: 0.4454  

Accuracy by scenario:  
  Basketball: 0.6000 (55 samples)  
  Bouldering: 0.2532 (79 samples)  
  Cooking: 0.5000 (12 samples)  
  Dance: 0.4490 (49 samples)  
  Music: 0.6786 (28 samples)  
  Soccer: 0.3333 (6 samples)  

Overall predicted class distribution:  
  Class 0: 35 samples (15.28%)  
  Class 1: 91 samples (39.74%)  
  Class 2: 41 samples (17.90%)  
  Class 3: 62 samples (27.07%)  

In [1]:
import json
from pathlib import Path

# This notebook runs from idea_1/ (the cells above load ./TimeSformer and ./pretrained),
# but the dataset sits at the repo root, so plain "data/..." does not resolve here.
# Walk up until we find it, which works from idea_1/ or from the repo root.
DATA_ROOT = next(
    p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "annotations").is_dir()
)
ANNOTATION_ROOT = DATA_ROOT / "annotations"
print(f"DATA_ROOT: {DATA_ROOT}")

with open(ANNOTATION_ROOT / "proficiency_demonstrator_train.json", "r") as file:
    proficiency_metadata = json.load(file)["annotations"]

with open(ANNOTATION_ROOT / "proficiency_demonstrator_val.json", "r") as file:
    proficiency_metadata_val = json.load(file)["annotations"]

print(f"train annotations: {len(proficiency_metadata)}, val annotations: {len(proficiency_metadata_val)}")

DATA_ROOT: D:\Multimodal-Ego-Exo4D\data
train annotations: 1904, val annotations: 466


In [2]:
import re
from torch.utils.data import Dataset, DataLoader

LEVELS = {
    "Novice": 1, 
    "Early Expert": 2, 
    "Intermediate Expert": 3, 
    "Late Expert": 4,
}

# proficiency_demonstrator_*.json exposes video_paths under these keys only -- there is
# no plain "exo", so asking for one is a KeyError. cam01 (exo1) is the usual exo baseline.
VIDEO_TYPES = ("ego", "exo1", "exo2", "exo3", "exo4")

# Expert commentary lives at <COMMENTARY_ROOT>/<take_name>/<expert_id>/data.json, with the
# transcribed text in a transcriptions.json next to each data.json.
COMMENTARY_ROOT = ANNOTATION_ROOT / "expert_commentary"

# Experts sometimes say the label outright ("this is a novice"); mask it so the
# model can't just read the answer off the commentary.
RATING_TERMS = re.compile(
    r"\b(?:novice|early\s+expert|intermediate\s+expert|late\s+expert|"
    r"proficien\w*|rating|score|good\s+execution|tip\s+for\s+improvement|beginner)\b",
    flags=re.IGNORECASE,
)

def take_name(m):
    """The take folder name, e.g. uniandes_bouldering_028_7.

    These annotations carry no take_name field, so read it off the video path
    ("takes/<take_name>/frame_aligned_videos/..."). takes.json maps take_uid ->
    take_name too, but it is missing ~250 of these uids.
    """
    return m["video_paths"]["ego"].split("/")[1]

def video_path(m, video_type):
    """Absolute path to the downscaled 448p video; only downscaled copies are on disk."""
    parts = m["video_paths"][video_type].split("/")
    parts[-1:-1] = ["downscaled", "448"]
    return DATA_ROOT.joinpath(*parts)

def load_commentary(take):
    """Return the take's expert commentary grouped per expert, or None if there is none."""
    experts = []
    for data_path in sorted((COMMENTARY_ROOT / take).rglob("data.json")):
        transcript_path = data_path.parent / "transcriptions.json"
        if not transcript_path.exists():
            continue
        data = json.loads(data_path.read_text(encoding="utf-8"))
        transcripts = json.loads(transcript_path.read_text(encoding="utf-8"))
        segments = []
        for annotation in data.get("annotations", []):
            text = transcripts.get(annotation.get("recording_path"), {}).get("text", "").strip()
            if text:
                segments.append((annotation.get("video_time") or 0.0, RATING_TERMS.sub("[MASK]", text)))
        if segments:
            # Most takes are narrated by 2-7 independent experts. Keep them in separate
            # blocks: merging them into one timestamp-sorted stream reads as a single
            # speaker talking over themselves.
            lines = "\n".join(f"[{t:.1f}s] {text}" for t, text in sorted(segments))
            experts.append(f"Expert {len(experts) + 1}:\n{lines}")
    if not experts:
        return None
    return "\n\n".join(experts)

class PromptDataset(Dataset):
    def __init__(self, proficiency_metadata, video_type='ego', require_commentary=True):
        if video_type not in VIDEO_TYPES:
            raise ValueError(f"video_type must be one of {VIDEO_TYPES}, got {video_type!r}")
        if not COMMENTARY_ROOT.exists():
            raise FileNotFoundError(f"{COMMENTARY_ROOT} not found; point COMMENTARY_ROOT at the expert_commentary folder")
        self.paths, self.commentary, self.targets = [], [], []
        self.take_names, self.scenarios = [], []
        self.dropped_no_video = self.dropped_no_commentary = 0
        for m in proficiency_metadata:
            path = video_path(m, video_type)
            if not path.exists():  # not every annotated take has been downloaded
                self.dropped_no_video += 1
                continue
            commentary = load_commentary(take_name(m))
            if commentary is None and require_commentary:
                self.dropped_no_commentary += 1
                continue
            self.paths.append(str(path))
            self.commentary.append(commentary or "(no commentary available)")
            self.targets.append(LEVELS[m['proficiency_score']])
            self.take_names.append(take_name(m))
            self.scenarios.append(m['scenario_name'])

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [
                {"type": "video", "video": self.paths[idx]},
                {"type": "text", "text": f"Expert commentary:\n{self.commentary[idx]}"},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": f"Score: {self.targets[idx]}"}]},
        ]
        return {"messages": messages, "score": self.targets[idx]}

system_prompt = """You assess the proficiency of a performer from a video and an expert's commentary on that video.

Base your judgment on what is observable in the video. Use the commentary as supporting evidence about technique, not as a substitute for watching.

Proficiency scale:
1 - Novice
2 - Early Expert
3 - Intermediate Expert
4 - Late Expert

Respond in exactly this format and nothing else:
Score: <single digit 1-4>
"""

train_dataset = PromptDataset(proficiency_metadata)
val_dataset = PromptDataset(proficiency_metadata_val)
print(f"train: {len(train_dataset)} takes, val: {len(val_dataset)} takes")
for name, ds in [("train", train_dataset), ("val", val_dataset)]:
    print(f"  {name} dropped: {ds.dropped_no_video} without video, {ds.dropped_no_commentary} without commentary")

train: 1174 takes, val: 334 takes
  train dropped: 249 without video, 481 without commentary
  val dropped: 10 without video, 122 without commentary


In [3]:
import torch
import shutil
import warnings
from transformers import AutoModelForMultimodalLM, AutoProcessor, Trainer, TrainingArguments, BitsAndBytesConfig
from transformers import TrainerCallback, TrainerControl, TrainerState
from transformers.video_utils import load_video
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen3.5-4B"
NUM_FRAMES = 16  # frames sampled per video; more frames = more visual tokens

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
# Left padding puts every answer at the very end of its row, so collate_fn can ask the
# model for logits on just those last few positions (see logits_to_keep below).
processor.tokenizer.padding_side = "left"

# transformers always decodes videos with torchcodec when it is installed, but torchcodec
# needs FFmpeg's shared DLLs, which this machine doesn't have ("Could not load libtorchcodec").
# Decode with PyAV instead -- it bundles its own FFmpeg.
def fetch_videos_pyav(video_url_or_urls, sample_indices_fn=None):
    if isinstance(video_url_or_urls, list):
        return list(zip(*[fetch_videos_pyav(x, sample_indices_fn) for x in video_url_or_urls]))
    return load_video(video_url_or_urls, backend="pyav", sample_indices_fn=sample_indices_fn)

processor.video_processor.fetch_videos = fetch_videos_pyav
# Qwen3-VL samples at fps=2 by default and rejects num_frames alongside it; we want a fixed NUM_FRAMES.
processor.video_processor.fps = None

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Only the assistant's answer contributes to the loss: mask everything up to and
# including the last "<|im_start|>assistant\n" header, plus padding.
ASSISTANT_HEADER = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

# Commentary is ~2.4k tokens per take at the median but up to ~21k, which on its own can
# exhaust 12 GB during the backward pass. Keep the first MAX_COMMENTARY_TOKENS tokens
# (covers ~90% of takes untouched).
MAX_COMMENTARY_TOKENS = 6000

def truncate_commentary(messages):
    text_block = next(b for b in messages[1]["content"] if b["type"] == "text")
    ids = processor.tokenizer.encode(text_block["text"], add_special_tokens=False)
    if len(ids) > MAX_COMMENTARY_TOKENS:
        text_block["text"] = processor.tokenizer.decode(ids[:MAX_COMMENTARY_TOKENS]) + "\n[commentary truncated]"
    return messages

def collate_fn(examples):
    batch = processor.apply_chat_template(
        [truncate_commentary(ex["messages"]) for ex in examples],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"padding": True, "num_frames": NUM_FRAMES},
    )
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    n = len(ASSISTANT_HEADER)
    for i, ids in enumerate(batch["input_ids"].tolist()):
        starts = [j for j in range(len(ids) - n + 1) if ids[j:j + n] == ASSISTANT_HEADER]
        labels[i, :starts[-1] + n] = -100
    # By default the model computes logits over the 248k-token vocab at every position and
    # upcasts them to fp32 for the loss -- ~9 GB for two 5k-token prompts -- although only the
    # ~12 answer tokens are supervised. Keep logits (and labels) from the position that
    # predicts the earliest answer token onward; left padding makes that a short tail.
    first_target = (labels != -100).int().argmax(dim=1).min().item()
    keep = labels.shape[1] - first_target + 1
    batch["labels"] = labels[:, -keep:]
    batch["logits_to_keep"] = keep
    return batch

training_args = TrainingArguments(
    output_dir="./qwen35-4b-finetuned",
    per_device_train_batch_size=1,  # prompts run 3k-8k tokens; one per step keeps peak memory down
    gradient_accumulation_steps=8,  # same effective batch of 8
    learning_rate=2e-4,
    bf16=True,  # Recommended over fp16 for native Qwen3.5 stability
    logging_steps=10,
    max_steps=100,  # Adjust to your dataset size/epochs
    save_strategy="steps",
    save_steps=10,  # a crash costs at most 10 steps
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=1,
    # Not paged_adamw_8bit: paged optimizers keep their state in CUDA unified memory, and reading it
    # back to save optimizer.pt crashed the NVIDIA driver on Windows (bugcheck 0x1E in nvlddmkm.sys,
    # unified-memory fault) at both step 50 and step 60. Same 8-bit AdamW, ordinary GPU memory.
    optim="adamw_8bit",
    gradient_checkpointing=True,
    remove_unused_columns=False,  # keep "messages" for collate_fn
)

# Resuming. A checkpoint is complete only once trainer_state.json (written last) exists.
# checkpoint-50 and -60 were cut off mid-save when the machine crashed: the adapter is intact, but
# optimizer.pt is truncated and trainer_state.json / scheduler.pt / rng_state.pth are missing,
# so resume_from_checkpoint can't use it directly. repair_checkpoint() rebuilds a resumable copy.
RESUME_FROM = "./qwen35-4b-finetuned/checkpoint-60"  # None to train from scratch

def repair_checkpoint(checkpoint):
    """Return a resumable checkpoint dir, rebuilding the trainer state if the save was cut off.

    The copy holds the adapter plus a reconstructed trainer_state.json, so the Trainer reloads the
    adapter, resumes at the same global step and skips the samples already trained on (same seed,
    so same shuffle). Without a usable optimizer.pt, Adam's moment estimates restart from zero.
    """
    checkpoint = Path(checkpoint)
    if (checkpoint / "trainer_state.json").exists():
        return str(checkpoint)
    repaired = checkpoint.with_name(checkpoint.name + "-repaired")
    repaired.mkdir(exist_ok=True)
    for name in ("adapter_model.safetensors", "adapter_config.json", "README.md"):
        shutil.copy2(checkpoint / name, repaired / name)
    global_step = int(checkpoint.name.rsplit("-", 1)[-1])
    samples_seen = global_step * training_args.gradient_accumulation_steps * training_args.per_device_train_batch_size
    TrainerState(
        global_step=global_step,
        epoch=samples_seen / len(train_dataset),
        logging_steps=training_args.logging_steps,
        eval_steps=training_args.eval_steps,
        save_steps=training_args.save_steps,
        train_batch_size=training_args.per_device_train_batch_size,
        stateful_callbacks=[TrainerControl()],  # the Trainer reads this back when saving the next checkpoint
    ).save_to_json(str(repaired / "trainer_state.json"))
    print(f"Rebuilt {repaired} from {checkpoint} (global step {global_step}); optimizer state starts fresh")
    return str(repaired)

class FastForwardLRSchedule(TrainerCallback):
    """Without scheduler.pt the Trainer starts a fresh LR schedule at step 0. Advance it to the
    resumed global step so the LR continues where the interrupted run left off (no-op otherwise)."""
    def on_train_begin(self, args, state, control, lr_scheduler=None, **kwargs):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # "lr_scheduler.step() before optimizer.step()" is intended here
            for _ in range(state.global_step - lr_scheduler.last_epoch):
                lr_scheduler.step()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    callbacks=[FastForwardLRSchedule()],
)


W1024 14:10:31.452000 32764 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

trainable params: 21,233,664 || all params: 4,560,499,200 || trainable%: 0.4656


In [ ]:
# Train (or resume). Skip this cell to only evaluate a checkpoint below.
trainer.train(resume_from_checkpoint=repair_checkpoint(RESUME_FROM) if RESUME_FROM else None)

trainer.model.save_pretrained("./qwen35-4b-adapter")

In [4]:
import contextlib
import csv
from collections import Counter, defaultdict
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
from tqdm.auto import tqdm

SCORE_TOKENS = processor.tokenizer.convert_tokens_to_ids([str(s) for s in LEVELS.values()])
# Video-only ablation: the same placeholder the dataset uses for takes without commentary, so the
# prompt keeps the format the model was trained on and only the commentary itself is missing.
# Commentary-only ablation: the video block is dropped from the prompt, so no frames are sent.
NO_COMMENTARY = "Expert commentary:\n(no commentary available)"
MODES = {(True, True): "video + commentary", (False, True): "video only", (True, False): "commentary only"}

@torch.inference_mode()
def predict(example, use_commentary=True, use_video=True):
    """Probabilities for scores 1-4 from one forward pass, formatted exactly as in training.

    The logits at the position just before the score digit are the model's prediction for it. The
    digit comes after that position, so causal attention never lets it influence the prediction;
    it's filled with a placeholder regardless.
    """
    user_content = example["messages"][1]["content"]
    if not use_commentary:
        next(b for b in user_content if b["type"] == "text")["text"] = NO_COMMENTARY
    if not use_video:
        example["messages"][1]["content"] = [b for b in user_content if b["type"] != "video"]
    example["messages"][-1]["content"][0]["text"] = "Score: 1"
    batch = collate_fn([example])
    batch.pop("labels"), batch.pop("logits_to_keep")
    ids = batch["input_ids"][0].tolist()
    digit_pos = max(j for j, t in enumerate(ids) if t == SCORE_TOKENS[0])  # the placeholder; only <|im_end|> follows
    assert processor.tokenizer.decode(ids[:digit_pos]).endswith("Score: "), "unexpected prompt layout"
    assert ("pixel_values_videos" in batch) == use_video
    with torch.autocast("cuda", dtype=torch.bfloat16):  # training ran under bf16 autocast too
        logits = model(**batch.to(model.device), logits_to_keep=len(ids) - digit_pos + 1).logits[0, 0]
    return logits[SCORE_TOKENS].float().softmax(-1).tolist()

def evaluate(checkpoint, split="test", use_commentary=True, use_video=True):
    """Score val takes with Qwen (a checkpoint's adapter, or zero-shot) and print a report.

    Only adapter_model.safetensors is read, so checkpoints whose optimizer.pt was cut off by the
    crashes work. split="test" scores the test split: the second half of the val annotations, listed
    in SkillFormer/annotation_test.jsonl, so it is the same test set the baselines use; Qwen only
    used val to log its loss, never to pick a checkpoint. split="val" scores every val take. Either
    way only takes with commentary are in val_dataset, so with and without commentary score the
    same takes. Predictions are appended to a CSV as they finish; re-running resumes from it.

    checkpoint=None evaluates the non-fine-tuned model zero-shot: the same 4-bit Qwen3.5-4B, prompt and
    scoring, with the LoRA adapter disabled.
    """
    if checkpoint is not None:
        load_result = set_peft_model_state_dict(model, load_file(Path(checkpoint) / "adapter_model.safetensors"))
        assert not load_result.unexpected_keys, load_result.unexpected_keys[:5]
    model.eval()

    if split == "test":
        with open("SkillFormer/annotation_test.jsonl", encoding="utf-8") as file:
            test_takes = {json.loads(line)["video_paths"][0].split("/")[1] for line in file}
        eval_indices = [i for i, take in enumerate(val_dataset.take_names) if take in test_takes]
    else:
        eval_indices = list(range(len(val_dataset)))

    mode = MODES[use_commentary, use_video]
    suffix = "" if use_commentary and use_video else "_" + mode.replace(" ", "_")
    name = Path(checkpoint).name if checkpoint is not None else "base"
    results_path = Path(f"eval_{name}_{split}{suffix}.csv")
    fields = ["take_name", "scenario", "true", "pred", "p1", "p2", "p3", "p4"]
    done = {}
    if results_path.exists():
        with open(results_path, newline="", encoding="utf-8") as file:
            done = {row["take_name"]: row for row in csv.DictReader(file)}
    todo = [i for i in eval_indices if val_dataset.take_names[i] not in done]
    label = checkpoint if checkpoint is not None else "base Qwen3.5-4B (zero-shot)"
    print(f"{label} on {split}, {mode}: {len(eval_indices)} takes ({len(eval_indices) - len(todo)} already scored)")

    adapter = model.disable_adapter() if checkpoint is None else contextlib.nullcontext()
    with adapter, open(results_path, "a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        if not done:
            writer.writeheader()
        for i in tqdm(todo):
            probs = predict(val_dataset[i], use_commentary, use_video)
            row = {"take_name": val_dataset.take_names[i], "scenario": val_dataset.scenarios[i],
                   "true": val_dataset.targets[i], "pred": 1 + max(range(4), key=probs.__getitem__),
                   **{f"p{s}": f"{p:.4f}" for s, p in zip(LEVELS.values(), probs)}}
            writer.writerow(row)
            file.flush()
            done[row["take_name"]] = row

    # Same report format as the SkillFormer results above (class k = score k + 1).
    rows = [done[val_dataset.take_names[i]] for i in eval_indices]
    correct = [int(r["pred"]) == int(r["true"]) for r in rows]
    by_scenario = defaultdict(list)
    for r, ok in zip(rows, correct):
        by_scenario[r["scenario"]].append(ok)
    print(f"\nOverall test accuracy: {sum(correct) / len(rows):.4f}\n\nAccuracy by scenario:")
    for scenario, oks in sorted(by_scenario.items()):
        print(f"  {scenario}: {sum(oks) / len(oks):.4f} ({len(oks)} samples)")
    print("\nOverall predicted class distribution:")
    pred_counts = Counter(int(r["pred"]) for r in rows)
    for level, score in LEVELS.items():
        print(f"  Class {score - 1} ({level}): {pred_counts[score]} samples ({pred_counts[score] / len(rows):.2%})")
    majority = Counter(int(r["true"]) for r in rows).most_common(1)[0]
    print(f"\nMajority-class baseline (always predict class {majority[0] - 1}): {majority[1] / len(rows):.4f}")
    return rows

In [5]:
# Checkpoint 50, video + expert commentary (the setting it was trained on)
rows_with_commentary = evaluate("./qwen35-4b-finetuned/checkpoint-50")

./qwen35-4b-finetuned/checkpoint-50 on test, video + commentary: 164 takes (164 already scored)


0it [00:00, ?it/s]


Overall test accuracy: 0.4024

Accuracy by scenario:
  Basketball: 0.4524 (42 samples)
  Cooking: 0.2500 (12 samples)
  Dance: 0.2895 (38 samples)
  Music: 0.8125 (16 samples)
  Rock Climbing: 0.3725 (51 samples)
  Soccer: 0.2000 (5 samples)

Overall predicted class distribution:
  Class 0 (Novice): 1 samples (0.61%)
  Class 1 (Early Expert): 109 samples (66.46%)
  Class 2 (Intermediate Expert): 27 samples (16.46%)
  Class 3 (Late Expert): 27 samples (16.46%)

Majority-class baseline (always predict class 3): 0.3537


In [6]:
# Checkpoint 50, video only: same takes, commentary replaced by the no-commentary placeholder
rows_video_only = evaluate("./qwen35-4b-finetuned/checkpoint-50", use_commentary=False)

changed = sum(int(a["pred"]) != int(b["pred"]) for a, b in zip(rows_with_commentary, rows_video_only))
print(f"\nPrediction changed on {changed}/{len(rows_video_only)} takes when the commentary was removed")

./qwen35-4b-finetuned/checkpoint-50 on test, video only: 164 takes (164 already scored)


0it [00:00, ?it/s]


Overall test accuracy: 0.2988

Accuracy by scenario:
  Basketball: 0.2381 (42 samples)
  Cooking: 0.1667 (12 samples)
  Dance: 0.2895 (38 samples)
  Music: 0.1875 (16 samples)
  Rock Climbing: 0.3922 (51 samples)
  Soccer: 0.6000 (5 samples)

Overall predicted class distribution:
  Class 0 (Novice): 0 samples (0.00%)
  Class 1 (Early Expert): 121 samples (73.78%)
  Class 2 (Intermediate Expert): 38 samples (23.17%)
  Class 3 (Late Expert): 5 samples (3.05%)

Majority-class baseline (always predict class 3): 0.3537

Prediction changed on 60/164 takes when the commentary was removed


In [7]:
# Checkpoint 50, commentary only: same takes, video block removed from the prompt (no frames)
rows_commentary_only = evaluate("./qwen35-4b-finetuned/checkpoint-50", use_video=False)

# The three settings side by side, then the same table in LaTeX (booktabs; best per row in bold)
settings = {"Video + commentary": rows_with_commentary, "Video only": rows_video_only,
            "Commentary only": rows_commentary_only}

def accuracy(rows):
    return sum(int(r["pred"]) == int(r["true"]) for r in rows) / len(rows)

table = [("Overall", len(rows_with_commentary), [accuracy(rows) for rows in settings.values()])]
for scenario in sorted({r["scenario"] for r in rows_with_commentary}):
    subsets = [[r for r in rows if r["scenario"] == scenario] for rows in settings.values()]
    table.append((scenario, len(subsets[0]), [accuracy(rows) for rows in subsets]))
majority_count = Counter(int(r["true"]) for r in rows_with_commentary).most_common(1)[0][1]
majority = majority_count / len(rows_with_commentary)

print(f"\n{'Scenario':<16}{'n':>5}" + "".join(f"{name:>22}" for name in settings))
for name, n, accs in table:
    print(f"{name:<16}{n:>5}" + "".join(f"{a:>22.4f}" for a in accs))
print(f"{'Majority class':<16}{'':>5}{majority:>22.4f}  (always predict the most common level)")

def cell(a, accs):
    return rf"\textbf{{{a:.3f}}}" if a == max(accs) else f"{a:.3f}"

latex = [r"\begin{table}[t]", r"\centering",
         r"\caption{Proficiency estimation accuracy of Qwen3.5-4B fine-tuned with LoRA (checkpoint 50) "
         rf"on the {len(rows_with_commentary)} test takes with expert commentary, by input modality.}}",
         r"\label{tab:modality-ablation}", r"\begin{tabular}{lrccc}", r"\toprule",
         r"Scenario & $n$ & " + " & ".join(settings) + r" \\", r"\midrule"]
for k, (name, n, accs) in enumerate(table):
    latex.append(f"{name} & {n} & " + " & ".join(cell(a, accs) for a in accs) + r" \\")
    if k == 0:
        latex.append(r"\midrule")
latex += [r"\midrule", rf"Majority class & & \multicolumn{{3}}{{c}}{{{majority:.3f}}} \\",
          r"\bottomrule", r"\end{tabular}", r"\end{table}"]
print("\n" + "\n".join(latex))

./qwen35-4b-finetuned/checkpoint-50 on test, commentary only: 164 takes (164 already scored)

0it [00:00, ?it/s]


Overall test accuracy: 0.4207

Accuracy by scenario:
  Basketball: 0.4524 (42 samples)
  Cooking: 0.3333 (12 samples)
  Dance: 0.3421 (38 samples)
  Music: 0.8125 (16 samples)
  Rock Climbing: 0.3725 (51 samples)
  Soccer: 0.2000 (5 samples)

Overall predicted class distribution:
  Class 0 (Novice): 0 samples (0.00%)
  Class 1 (Early Expert): 105 samples (64.02%)
  Class 2 (Intermediate Expert): 37 samples (22.56%)
  Class 3 (Late Expert): 22 samples (13.41%)

Majority-class baseline (always predict class 3): 0.3537

Scenario            n    Video + commentary            Video only       Commentary only
Overall           164                0.4024                0.2988                0.4207
Basketball         42                0.4524                0.2381                0.4524
Cooking            12                0.2500                0.1667                0.3333
Dance              38                0.2895                0.2895                0.3421
Music              16               

In [8]:
# Non-fine-tuned Qwen3.5-4B, zero-shot: same model, prompt, takes and scoring, LoRA adapter disabled
rows_base = {
    "Video": evaluate(None, use_commentary=False),
    "Video + commentary": evaluate(None),
    "Commentary": evaluate(None, use_video=False),
}

base Qwen3.5-4B (zero-shot) on test, video only: 164 takes (164 already scored)


0it [00:00, ?it/s]


Overall test accuracy: 0.2561

Accuracy by scenario:
  Basketball: 0.2143 (42 samples)
  Cooking: 0.2500 (12 samples)
  Dance: 0.1842 (38 samples)
  Music: 0.2500 (16 samples)
  Rock Climbing: 0.3333 (51 samples)
  Soccer: 0.4000 (5 samples)

Overall predicted class distribution:
  Class 0 (Novice): 18 samples (10.98%)
  Class 1 (Early Expert): 95 samples (57.93%)
  Class 2 (Intermediate Expert): 51 samples (31.10%)
  Class 3 (Late Expert): 0 samples (0.00%)

Majority-class baseline (always predict class 3): 0.3537
base Qwen3.5-4B (zero-shot) on test, video + commentary: 164 takes (164 already scored)


0it [00:00, ?it/s]


Overall test accuracy: 0.3902

Accuracy by scenario:
  Basketball: 0.4048 (42 samples)
  Cooking: 0.1667 (12 samples)
  Dance: 0.3421 (38 samples)
  Music: 0.7500 (16 samples)
  Rock Climbing: 0.3725 (51 samples)
  Soccer: 0.2000 (5 samples)

Overall predicted class distribution:
  Class 0 (Novice): 2 samples (1.22%)
  Class 1 (Early Expert): 74 samples (45.12%)
  Class 2 (Intermediate Expert): 73 samples (44.51%)
  Class 3 (Late Expert): 15 samples (9.15%)

Majority-class baseline (always predict class 3): 0.3537
base Qwen3.5-4B (zero-shot) on test, commentary only: 164 takes (164 already scored)


0it [00:00, ?it/s]


Overall test accuracy: 0.3476

Accuracy by scenario:
  Basketball: 0.4762 (42 samples)
  Cooking: 0.1667 (12 samples)
  Dance: 0.2368 (38 samples)
  Music: 0.6250 (16 samples)
  Rock Climbing: 0.2353 (51 samples)
  Soccer: 0.8000 (5 samples)

Overall predicted class distribution:
  Class 0 (Novice): 11 samples (6.71%)
  Class 1 (Early Expert): 29 samples (17.68%)
  Class 2 (Intermediate Expert): 80 samples (48.78%)
  Class 3 (Late Expert): 44 samples (26.83%)

Majority-class baseline (always predict class 3): 0.3537


In [9]:
# Zero-shot vs. fine-tuned (checkpoint 50) for every input, in one table and in LaTeX
# (booktabs; best per row in bold)
results = {("Zero-shot", input_name): rows for input_name, rows in rows_base.items()}
results.update({
    ("Fine-tuned", "Video"): rows_video_only,
    ("Fine-tuned", "Video + commentary"): rows_with_commentary,
    ("Fine-tuned", "Commentary"): rows_commentary_only,
})

def accuracy(rows):
    return sum(int(r["pred"]) == int(r["true"]) for r in rows) / len(rows)

reference = rows_with_commentary
table = [("Overall", len(reference), [accuracy(rows) for rows in results.values()])]
for scenario in sorted({r["scenario"] for r in reference}):
    subsets = [[r for r in rows if r["scenario"] == scenario] for rows in results.values()]
    table.append((scenario, len(subsets[0]), [accuracy(rows) for rows in subsets]))
majority = Counter(int(r["true"]) for r in reference).most_common(1)[0][1] / len(reference)

headers = [f"{model_name}: {input_name}" for model_name, input_name in results]
print(f"{'Scenario':<15}{'n':>4}" + "".join(f"{h:>31}" for h in headers))
for scenario, n, accs in table:
    print(f"{scenario:<15}{n:>4}" + "".join(f"{a:>31.4f}" for a in accs))
print(f"{'Majority class':<15}{'':>4}{majority:>31.4f}")
print(f"{'Predicted N/E/I/L':<19}" + "".join(
    f"{'/'.join(str(sum(int(r['pred']) == s for r in rows)) for s in LEVELS.values()):>31}" for rows in results.values()))

SHORT = {"Video": "V", "Video + commentary": "V+C", "Commentary": "C"}

def cell(a, accs):
    return rf"\textbf{{{a:.3f}}}" if a == max(accs) else f"{a:.3f}"

latex = [
    r"\begin{table}[t]", r"\centering", r"\small",
    r"\caption{Proficiency estimation accuracy of Qwen3.5-4B on the "
    rf"{len(reference)} test takes with expert commentary, zero-shot and fine-tuned with LoRA "
    r"(checkpoint 50), by input: V = video, C = expert commentary.}",
    r"\label{tab:zero-shot-vs-finetuned}",
    r"\begin{tabular}{lrcccccc}", r"\toprule",
    r" & & \multicolumn{3}{c}{Zero-shot} & \multicolumn{3}{c}{Fine-tuned} \\",
    r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
    r"Scenario & $n$ & " + " & ".join(SHORT[input_name] for _, input_name in results) + r" \\",
    r"\midrule",
]
for k, (scenario, n, accs) in enumerate(table):
    latex.append(f"{scenario} & {n} & " + " & ".join(cell(a, accs) for a in accs) + r" \\")
    if k == 0:
        latex.append(r"\midrule")
latex += [r"\midrule", rf"Majority class & & \multicolumn{{6}}{{c}}{{{majority:.3f}}} \\",
          r"\bottomrule", r"\end{tabular}", r"\end{table}"]
print("\n" + "\n".join(latex))

Scenario          n               Zero-shot: Video  Zero-shot: Video + commentary          Zero-shot: Commentary              Fine-tuned: Video Fine-tuned: Video + commentary         Fine-tuned: Commentary
Overall         164                         0.2561                         0.3902                         0.3476                         0.2988                         0.4024                         0.4207
Basketball       42                         0.2143                         0.4048                         0.4762                         0.2381                         0.4524                         0.4524
Cooking          12                         0.2500                         0.1667                         0.1667                         0.1667                         0.2500                         0.3333
Dance            38                         0.1842                         0.3421                         0.2368                         0.2895                         0.2895  